# 实验一：环境认知与云环境体验——CANNLAB预置环境检查与华为云沙箱NPU验证

## 小节概述

完成 CANNLab 预置环境检查、华为云沙箱 NPU 验证和算子工程脚手架认知。

## 教程具体内容

以下内容围绕实验背景、任务准备、关键步骤和实验总结展开。


建议学时：2学时

## 实验任务

### 任务描述

先找到云开发环境并连接，地址：https://gitcode.com/org/cann/cannlab/environment

操作步骤：

1. 打开 CANNLab 云开发环境页面。
2. 选择课程验证所需的外部开发者可创建环境。
3. 记录 NPU 镜像模板名称和打开后的 Python 内核版本。
4. 点击连接并进入云开发环境，确认 notebook 能够正常打开。


---

原实验设计：

要求学生在华为云沙箱从零开始安装CANN软件栈，审计环境部署过程中的"隐式依赖"摩擦力。

修改后的设计：

本实验分为两个阶段：

- 阶段一（CANNLAB）：利用预置环境快速认知算子工程脚手架，理解开发工具链组成

- 阶段二（华为云沙箱）：在真实昇腾云环境下完成NPU设备验证和基础运行测试

修改原因：

环境部署虽然是开发者体验的重要组成，但对于算子开发课程而言，不应成为主要学习障碍。通过CANNLAB预置环境，学生可以快速进入算子开发核心环节，同时在华为云沙箱体验真实NPU环境，兼顾学习效率和实践真实性。

---

### 学习目标

小组分工：

- 开源软件开发工程师：在CANNLAB完成环境检查和脚手架认知，在华为云沙箱完成NPU验证

- 开源社区布道师：记录两个平台的使用体验差异，准备环境对比报告

- 开源合规与安全工程师：审计预置环境的工具链版本和依赖关系

学习资源路径：

- CANNLAB平台：访问CANNLAB在线实验平台，获取预配置的算子开发环境

- 华为云沙箱：进入华为云沙箱实验室，搜索《基于昇腾弹性云服务器的人工智能应用开发实验》

- 审计工具：准备环境检查脚本和依赖关系分析工具

---

## 任务准备

### 前置知识

本实验需要提前学习以下相关知识：

- CANN异构计算架构基础知识
- Python编程基础
- Linux命令行操作基础

### 实验环境准备

本实验需要在以下环境中进行：

- **CANNLAB平台**：访问CANNLAB在线实验平台，获取预配置的算子开发环境
- **华为云沙箱**：进入华为云沙箱实验室，启动昇腾弹性云服务器

## 任务实施

### 实验要点

- 步骤一：登录 CANNLab 并检查预置环境
- 步骤二：运行最小 NPU 样例验证环境可用性
- 步骤三：使用 msOpGen 生成并认识算子工程脚手架
- 步骤四：记录当前环境的 NPU、驱动与系统信息
- 步骤五：按需在华为云沙箱复核环境
- 步骤六：对比两个平台的环境差异


### 关键步骤

阶段一：CANNLab预置环境检查与脚手架认知（建议用时：30分钟）

步骤一：登录CANNLab并检查预置环境

登录CANNLab平台，进入算子开发实验环境。以下命令从环境变量读取 CANN 安装目录，不假设固定安装路径：


In [ ]:
%%bash
set -euo pipefail

ASCEND_HOME_DIR="${ASCEND_HOME_PATH:-${ASCEND_HOME:-}}"
if [[ -z "${ASCEND_HOME_DIR}" ]]; then
  echo "ERROR: ASCEND_HOME_PATH 或 ASCEND_HOME 未设置，请先加载 CANN 环境。" >&2
  exit 1
fi

echo "=== CANN 安装目录 ==="
echo "${ASCEND_HOME_DIR}"

echo "=== CANN 版本 ==="
VERSION_FILE="$(find "${ASCEND_HOME_DIR}" -maxdepth 2 -type f -name version.info -print -quit)"
if [[ -n "${VERSION_FILE}" ]]; then
  cat "${VERSION_FILE}"
else
  echo "未在安装目录前两层发现 version.info，请以镜像说明和环境变量为准。"
fi

echo "=== Python 与关键包 ==="
python3 --version
python3 - <<'PY'
from importlib.util import find_spec

for package in ("numpy", "torch", "torch_npu"):
    status = "已安装" if find_spec(package) else "未安装"
    print(f"{package}: {status}")
PY

echo "=== 算子开发工具链 ==="
for tool in msopgen msopst msprof; do
  if command -v "${tool}" >/dev/null 2>&1; then
    printf '%-8s %s\n' "${tool}" "$(command -v "${tool}")"
  else
    printf '%-8s %s\n' "${tool}" "当前镜像未提供"
  fi
done

command -v msopgen >/dev/null 2>&1 || {
  echo "ERROR: 本实验需要 msopgen，请切换到 CANN 算子开发镜像。" >&2
  exit 1
}


预期结果：控制台显示当前 CANN 安装目录、Python 版本和工具路径。版本号以实际镜像为准；本实验要求 `msopgen` 可用，其他工具若未预装应如实记录。

步骤二：运行最小样例验证环境可用性

在CANNLAB环境中运行最小张量计算，验证 NPU 执行链路：


In [ ]:
import torch
import torch_npu

# 检查NPU设备是否可用
print("NPU available:", torch.npu.is_available())
print("NPU count:", torch.npu.device_count())

# 运行简单的张量计算测试
x = torch.randn(2, 3).npu()
y = torch.randn(2, 3).npu()
z = x + y
print("Test computation result shape:", z.shape)
print("Environment validation: PASSED")

步骤三：使用 msOpGen 生成并认识算子工程脚手架

下面先写出 VectorAdd 的算子原型，再由当前 CANN 环境中的 `msopgen` 生成工程。这样 Notebook 不依赖预先存在的 `~/samples` 目录：


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess

work_root = Path(".cann_course_work/01_01")
project_dir = work_root / "vector_add"
prototype_path = work_root / "vector_add.json"

work_root.mkdir(parents=True, exist_ok=True)
shutil.rmtree(project_dir, ignore_errors=True)

prototype = [{
    "op": "VectorAdd",
    "input_desc": [
        {
            "name": "x",
            "param_type": "required",
            "format": ["ND", "ND"],
            "type": ["float16", "float"]
        },
        {
            "name": "y",
            "param_type": "required",
            "format": ["ND", "ND"],
            "type": ["float16", "float"]
        }
    ],
    "output_desc": [{
        "name": "z",
        "param_type": "required",
        "format": ["ND", "ND"],
        "type": ["float16", "float"]
    }]
}]
prototype_path.write_text(
    json.dumps(prototype, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

subprocess.run([
    "msopgen", "gen",
    "-i", str(prototype_path),
    "-c", "ai_core-ascend910b1",
    "-lan", "cpp",
    "-out", str(project_dir),
], check=True)

required_names = {"op_host", "op_kernel", "CMakeLists.txt"}
generated_names = {path.name for path in project_dir.rglob("*")}
missing = sorted(required_names - generated_names)
if missing:
    raise RuntimeError(f"msopgen 工程缺少关键项: {missing}")

print(f"工程已生成: {project_dir.resolve()}")
for generated in sorted(project_dir.rglob("*")):
    if len(generated.relative_to(project_dir).parts) <= 2:
        suffix = "/" if generated.is_dir() else ""
        print(f"  {generated.relative_to(project_dir)}{suffix}")


审计要求：学生需在报告中说明：

1. `op_host` 负责算子注册、Shape 推导和 Tiling 等 Host 侧逻辑；
2. `op_kernel` 负责 Ascend C Kernel 核心计算；
3. `CMakeLists.txt` 等构建文件负责组织编译与打包；
4. 测试代码负责构造输入、运行算子并与基准结果比较；
5. 工程结构可能随 CANN 版本变化，报告应以本次 `msopgen` 的实际输出为准。

步骤四：记录当前环境的 NPU、驱动与系统信息

以下单元可直接在 CANNLab 执行。若某项信息由平台托管而不可见，单元会明确记录“不可用”，不会把预期值当成实测值：


In [ ]:
%%bash
set -u

echo "=== 系统信息 ==="
uname -a
cat /etc/os-release

echo "=== NPU 状态 ==="
if command -v npu-smi >/dev/null 2>&1; then
  if ! npu-smi info; then
    echo "WARNING: npu-smi 已安装，但当前会话无法读取设备状态。"
  fi
else
  echo "当前环境未提供 npu-smi。"
fi

echo "=== 驱动版本 ==="
DRIVER_VERSION=/usr/local/Ascend/driver/version.info
if [[ -r "${DRIVER_VERSION}" ]]; then
  cat "${DRIVER_VERSION}"
else
  echo "当前环境未暴露 ${DRIVER_VERSION}。"
fi


步骤五：按需在华为云沙箱复核环境

此步骤依赖课程外部的华为云沙箱，不属于 CANNLab 自动执行范围。只有在已经进入华为云沙箱时，才设置环境变量 `RUN_HUAWEI_CLOUD_SANDBOX=1` 并运行下面的单元；默认运行会给出明确的跳过说明：


In [ ]:
import os

if os.environ.get("RUN_HUAWEI_CLOUD_SANDBOX") != "1":
    print("SKIP: 当前按 CANNLab 流程执行；华为云沙箱复核需手动设置 RUN_HUAWEI_CLOUD_SANDBOX=1。")
else:
    import torch
    import torch_npu

    if not torch.npu.is_available():
        raise RuntimeError("已选择华为云沙箱复核，但当前 Python 会话无法访问 NPU。")

    device = torch.device("npu:0")
    probe = torch.ones(8, dtype=torch.float32, device=device)
    result = torch.add(probe, probe)
    torch.npu.synchronize()
    if not torch.allclose(result.cpu(), torch.full((8,), 2.0)):
        raise RuntimeError("华为云沙箱 NPU 验证结果不正确。")

    print("华为云沙箱 NPU 验证: PASSED")
    print("Device ID: 0")


步骤六：对比两个平台的环境差异

记录CANNLAB与华为云沙箱在以下方面的差异：

| 对比维度 | CANNLAB | 华为云沙箱 |
| --- | --- | --- |
| 环境配置 | 预置完成，开箱即用 | 需要初始化，更接近生产环境 |
| NPU访问 | 平台分配实验资源 | 真实NPU云实例 |
| 工具链 | 完整预装 | 需要确认版本和路径 |
| 适用场景 | 学习和开发 | 性能测试和验证 |

---

## 任务拓展

每个小组需提交《环境认知与验证报告.md》，内容包括：当前环境的命令与关键输出、CANNLAB 与华为云沙箱的差异、由 `msopgen` 实际生成的算子工程结构图，以及 `op_host`、`op_kernel`、测试代码和构建文件的职责说明。无法访问华为云沙箱时，应标注该部分未执行，不能使用预期结果替代实测结果。


## 实验总结

通过本次实验，完成了以下关键学习目标：

- 完成了CANNLAB预置环境检查与华为云沙箱NPU验证
- 理解了两个平台的差异和各自适用场景
- 认知了算子工程脚手架结构

## 课后实践

请整理一份环境认知与验证报告，说明两个平台的环境差异、工具链状态和 host/kernel/test 目录职责。

## 参考答案

运行下面的代码单元查看参考报告。环境版本和设备信息应以实际运行结果为准。


In [ ]:
!cat ./answer/01.01_answer.md